# 06: Report Generation

This notebook demonstrates the reporting capabilities of siege_utilities:

1. **PDF Reports** - Using ReportLab
2. **Charts & Tables** - Using ChartGenerator
3. **Client Branding** - Applying custom styles
4. **PowerPoint Generation** - PPTX output

## Prerequisites

```bash
pip install -e /path/to/siege_utilities
pip install reportlab python-pptx pillow
```

In [ ]:
# Imports
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np

import siege_utilities as su
su.configure_shared_logging(level="INFO")

# Check for required dependencies
dependencies = {}

try:
    from reportlab.lib.pagesizes import letter
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
    from reportlab.lib.styles import getSampleStyleSheet
    from reportlab.lib import colors
    dependencies['reportlab'] = True
except ImportError:
    dependencies['reportlab'] = False

try:
    from pptx import Presentation
    from pptx.util import Inches, Pt
    dependencies['pptx'] = True
except ImportError:
    dependencies['pptx'] = False

try:
    from PIL import Image
    dependencies['pillow'] = True
except ImportError:
    dependencies['pillow'] = False

su.log_info("Dependency Status:")
for dep, available in dependencies.items():
    status = 'Available' if available else 'NOT INSTALLED'
    su.log_info(f"  {dep}: {status}")

In [ ]:
# Import siege_utilities components
from siege_utilities.reporting.chart_generator import ChartGenerator
from siege_utilities.reporting.templates.base_template import BaseReportTemplate

su.log_info("siege_utilities reporting imports successful!")

## 1. Create Sample Data

We'll create sample data for our reports.

In [ ]:
np.random.seed(42)

# Sample sales data
sales_data = pd.DataFrame({
    'Month': ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun'],
    'Revenue': [45000, 52000, 48000, 61000, 55000, 67000],
    'Expenses': [32000, 35000, 33000, 38000, 36000, 40000],
    'Units Sold': [150, 175, 160, 205, 185, 225]
})
sales_data['Profit'] = sales_data['Revenue'] - sales_data['Expenses']

su.log_info("Sales Data:")
su.log_info(f"\n{sales_data}")

In [ ]:
# Sample regional data
regional_data = pd.DataFrame({
    'Region': ['North', 'South', 'East', 'West', 'Central'],
    'Sales': [125000, 98000, 145000, 112000, 89000],
    'Customers': [450, 380, 520, 410, 340],
    'Avg Order': [278, 258, 279, 273, 262]
})

su.log_info("Regional Data:")
su.log_info(f"\n{regional_data}")

## 2. Basic PDF Generation with ReportLab

Create a simple PDF report using ReportLab directly.

In [ ]:
if dependencies['reportlab']:
    def create_simple_pdf(filename, title, data_df):
        """Create a simple PDF with a title and table."""
        doc = SimpleDocTemplate(filename, pagesize=letter)
        styles = getSampleStyleSheet()
        elements = []
        
        # Add title
        elements.append(Paragraph(title, styles['Title']))
        elements.append(Spacer(1, 12))
        
        # Add description
        elements.append(Paragraph("This report was generated using siege_utilities.", styles['Normal']))
        elements.append(Spacer(1, 24))
        
        # Convert DataFrame to table data
        table_data = [data_df.columns.tolist()] + data_df.values.tolist()
        
        # Create table
        table = Table(table_data)
        table.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
            ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('FONTSIZE', (0, 0), (-1, 0), 12),
            ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
            ('BACKGROUND', (0, 1), (-1, -1), colors.beige),
            ('TEXTCOLOR', (0, 1), (-1, -1), colors.black),
            ('FONTNAME', (0, 1), (-1, -1), 'Helvetica'),
            ('FONTSIZE', (0, 1), (-1, -1), 10),
            ('GRID', (0, 0), (-1, -1), 1, colors.black),
            ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ]))
        
        elements.append(table)
        
        # Build PDF
        doc.build(elements)
        return filename
    
    # Create sample PDF
    output_path = '/tmp/sample_sales_report.pdf'
    create_simple_pdf(output_path, 'Sales Report - Q1 2026', sales_data)
    su.log_info(f"PDF created: {output_path}")
else:
    su.log_warning("ReportLab not installed. Run: pip install reportlab")

## 3. Using ChartGenerator

Create charts for reports using the ChartGenerator class.

In [ ]:
# Initialize ChartGenerator
chart_gen = ChartGenerator()

su.log_info("ChartGenerator methods:")
chart_methods = [m for m in dir(chart_gen) if m.startswith('create_') and not m.startswith('_')]
for method in chart_methods[:10]:  # Show first 10
    su.log_info(f"  - {method}")

In [ ]:
# Create a bar chart
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(sales_data))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], sales_data['Revenue'], width, label='Revenue', color='#2ecc71')
bars2 = ax.bar([i + width/2 for i in x], sales_data['Expenses'], width, label='Expenses', color='#e74c3c')

ax.set_xlabel('Month')
ax.set_ylabel('Amount ($)')
ax.set_title('Monthly Revenue vs Expenses')
ax.set_xticks(x)
ax.set_xticklabels(sales_data['Month'])
ax.legend()

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'${height:,.0f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)

plt.tight_layout()
chart_path = '/tmp/revenue_expenses_chart.png'
fig.savefig(chart_path, dpi=150, bbox_inches='tight')
su.log_info(f"Chart saved: {chart_path}")
plt.close()

## 4. Using BaseReportTemplate

Create a formatted report using the siege_utilities template system.

In [ ]:
if dependencies['reportlab']:
    # Create a report using BaseReportTemplate
    output_file = '/tmp/formatted_report.pdf'
    
    try:
        template = BaseReportTemplate(
            output_filename=output_file,
            page_size='letter',
            margins={'top': 1.0, 'bottom': 1.0, 'left': 1.0, 'right': 1.0}
        )
        
        su.log_info("BaseReportTemplate initialized successfully")
        su.log_info(f"Available styles: {list(template.styles.byName.keys())[:5]}...")
        
    except Exception as e:
        su.log_error(f"Error initializing template: {e}")

## 5. PDF Report with Table and Chart

Create a complete report combining text, tables, and charts.

In [ ]:
if dependencies['reportlab']:
    from reportlab.platypus import Image as RLImage
    from reportlab.lib.units import inch
    
    def create_complete_report(filename, title, data_df, chart_image_path=None):
        """Create a complete PDF report with title, table, and optional chart."""
        doc = SimpleDocTemplate(filename, pagesize=letter)
        styles = getSampleStyleSheet()
        elements = []
        
        # Title
        elements.append(Paragraph(title, styles['Title']))
        elements.append(Spacer(1, 12))
        
        # Report date
        from datetime import datetime
        date_str = datetime.now().strftime('%B %d, %Y')
        elements.append(Paragraph(f"Report Date: {date_str}", styles['Normal']))
        elements.append(Spacer(1, 24))
        
        # Executive Summary
        elements.append(Paragraph("Executive Summary", styles['Heading2']))
        summary = """This report provides an analysis of sales performance for the first 
        half of 2026. Key metrics include monthly revenue, expenses, and profit margins."""
        elements.append(Paragraph(summary, styles['Normal']))
        elements.append(Spacer(1, 12))
        
        # Data Table Section
        elements.append(Paragraph("Sales Data Summary", styles['Heading2']))
        elements.append(Spacer(1, 6))
        
        # Format numbers in DataFrame for display
        display_df = data_df.copy()
        for col in ['Revenue', 'Expenses', 'Profit']:
            if col in display_df.columns:
                display_df[col] = display_df[col].apply(lambda x: f'${x:,.0f}')
        
        table_data = [display_df.columns.tolist()] + display_df.values.tolist()
        
        table = Table(table_data, colWidths=[1*inch, 1.2*inch, 1.2*inch, 1*inch, 1.2*inch])
        table.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#2c3e50')),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
            ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('FONTSIZE', (0, 0), (-1, 0), 10),
            ('BOTTOMPADDING', (0, 0), (-1, 0), 10),
            ('TOPPADDING', (0, 0), (-1, 0), 10),
            ('BACKGROUND', (0, 1), (-1, -1), colors.HexColor('#ecf0f1')),
            ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor('#ecf0f1')]),
            ('TEXTCOLOR', (0, 1), (-1, -1), colors.black),
            ('FONTNAME', (0, 1), (-1, -1), 'Helvetica'),
            ('FONTSIZE', (0, 1), (-1, -1), 9),
            ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
            ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ]))
        
        elements.append(table)
        elements.append(Spacer(1, 24))
        
        # Chart Section
        if chart_image_path and Path(chart_image_path).exists():
            elements.append(Paragraph("Revenue vs Expenses Visualization", styles['Heading2']))
            elements.append(Spacer(1, 6))
            
            img = RLImage(chart_image_path, width=6*inch, height=3.6*inch)
            elements.append(img)
            elements.append(Spacer(1, 12))
        
        # Key Findings
        elements.append(Paragraph("Key Findings", styles['Heading2']))
        findings = [
            "Revenue showed consistent growth, with June achieving the highest at $67,000.",
            "Profit margins improved throughout the period, reaching 40% in June.",
            "Units sold increased by 50% from January to June."
        ]
        for finding in findings:
            bullet = Paragraph(f"\u2022 {finding}", styles['Normal'])
            elements.append(bullet)
            elements.append(Spacer(1, 6))
        
        # Build PDF
        doc.build(elements)
        return filename
    
    # Create the complete report
    output_path = '/tmp/complete_sales_report.pdf'
    create_complete_report(
        output_path, 
        'Sales Performance Report - H1 2026',
        sales_data,
        chart_image_path='/tmp/revenue_expenses_chart.png'
    )
    su.log_info(f"Complete report created: {output_path}")
else:
    su.log_warning("ReportLab not installed")

## 6. PowerPoint Generation (Optional)

Create PowerPoint presentations using python-pptx.

In [ ]:
if dependencies['pptx']:
    def create_pptx(filename, title, subtitle, data_df, chart_path=None):
        """Create a PowerPoint presentation."""
        prs = Presentation()
        
        # Title slide
        title_slide_layout = prs.slide_layouts[0]
        slide = prs.slides.add_slide(title_slide_layout)
        slide.shapes.title.text = title
        slide.placeholders[1].text = subtitle
        
        # Content slide with table
        table_slide_layout = prs.slide_layouts[5]  # Blank layout
        slide = prs.slides.add_slide(table_slide_layout)
        
        # Add title
        shapes = slide.shapes
        shapes.title.text = "Sales Summary"
        
        # Add table
        rows, cols = len(data_df) + 1, len(data_df.columns)
        left = Inches(0.5)
        top = Inches(1.5)
        width = Inches(9)
        height = Inches(0.8 * rows)
        
        table = shapes.add_table(rows, cols, left, top, width, height).table
        
        # Header row
        for i, col_name in enumerate(data_df.columns):
            table.cell(0, i).text = str(col_name)
        
        # Data rows
        for row_idx, row in data_df.iterrows():
            for col_idx, value in enumerate(row):
                table.cell(row_idx + 1, col_idx).text = str(value)
        
        # Chart slide (if chart exists)
        if chart_path and Path(chart_path).exists():
            chart_slide_layout = prs.slide_layouts[5]
            slide = prs.slides.add_slide(chart_slide_layout)
            
            # Add chart image
            left = Inches(1)
            top = Inches(1.5)
            slide.shapes.add_picture(chart_path, left, top, width=Inches(8))
        
        prs.save(filename)
        return filename
    
    # Create PowerPoint
    from datetime import datetime
    pptx_path = '/tmp/sales_presentation.pptx'
    create_pptx(
        pptx_path,
        'Sales Performance Review',
        f'H1 2026 | Generated {datetime.now().strftime("%Y-%m-%d")}',
        sales_data,
        chart_path='/tmp/revenue_expenses_chart.png'
    )
    su.log_info(f"PowerPoint created: {pptx_path}")
else:
    su.log_warning("python-pptx not installed. Run: pip install python-pptx")

## Summary

This notebook demonstrated:

| Feature | Description |
|---------|-------------|
| `SimpleDocTemplate` | Basic PDF generation with ReportLab |
| `Table` / `TableStyle` | Formatted data tables |
| `ChartGenerator` | Chart creation for reports |
| `BaseReportTemplate` | siege_utilities template system |
| `Presentation` | PowerPoint generation with python-pptx |

**Files Generated:**
- `/tmp/sample_sales_report.pdf` - Simple PDF with table
- `/tmp/complete_sales_report.pdf` - Full report with chart
- `/tmp/sales_presentation.pptx` - PowerPoint deck